# Phase 4A2 -- TaViT Evaluation: Temporal Trajectory Assessment

> **Purpose**: Evaluate TaViT trajectory embeddings (256-D per patient) against the
> static per-scan embeddings on temporal tests T1-T8, plus new trajectory-specific metrics.
>
> **Inputs**:
> - `tavit_trajectory_embeddings.npz` (from Phase4_A1)
> - `vit_swinunetr_embeddings_v5_hybrid.npz` (static baseline)
> - `tumor_volumes.csv`
>
> **No GPU required** -- pure numpy/sklearn/scipy.


In [ ]:
import numpy as np
import json as _json
import warnings
import os
from pathlib import Path
from collections import defaultdict

import pandas as pd
from scipy.stats import spearmanr, kendalltau, mannwhitneyu
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import RidgeClassifier, Ridge
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import (f1_score, roc_auc_score, cohen_kappa_score,
                             classification_report, silhouette_score)
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors

warnings.filterwarnings("ignore")
np.random.seed(42)

OUTPUT_ROOT = Path("/kaggle/working/tavit_evaluation")
FIG_DIR = OUTPUT_ROOT / "figures"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

SEARCH_ROOTS = [Path("/kaggle/input"), Path("/kaggle/working")]

def find_file(names):
    for root in SEARCH_ROOTS:
        if not root.exists():
            continue
        for f in root.rglob("*"):
            for name in names:
                if name in f.name:
                    return f
    return None

# ============================================================
# LOAD TAVIT TRAJECTORY EMBEDDINGS
# ============================================================
traj_path = find_file(["tavit_trajectory_embeddings.npz"])
assert traj_path is not None, "tavit_trajectory_embeddings.npz not found!"
traj_npz = np.load(traj_path, allow_pickle=True)
traj_pids = sorted(traj_npz.files)
print(f"TaViT trajectories: {len(traj_pids)} patients | "
      f"dim={traj_npz[traj_pids[0]].shape[0]}")
print(f"  File: {traj_path}")

# Load trajectory metadata if available
meta_path = find_file(["tavit_trajectory_meta.json"])
traj_meta = {}
if meta_path:
    with open(meta_path) as f:
        traj_meta = _json.load(f)
    print(f"  Metadata: {len(traj_meta)} entries")

# ============================================================
# ============================================================
# LOAD STATIC HYBRID EMBEDDINGS (for comparison)
# ============================================================
static_path = find_file(["vit_swinunetr_embeddings_v5_hybrid.npz",
                          "vit_swinunetr_embeddings_v5_noglobal.npz"])
static_npz = None
static_patient_seqs = {}
if static_path:
    raw_npz = np.load(static_path, allow_pickle=True)
    all_npz_keys = sorted(raw_npz.files)
    print(f"\nStatic hybrid NPZ keys ({len(all_npz_keys)}): {all_npz_keys[:4]}")
    print(f"  File: {static_path}")

    # Detect format: per-scan keys vs batch arrays
    def is_scan_key(k):
        return "__" in k and any(k.split("__")[1].startswith(p) for p in ["t","p"])

    format_a_keys = [k for k in all_npz_keys if is_scan_key(k)]
    if len(format_a_keys) >= len(all_npz_keys) * 0.5:
        # Format A: per-scan keys
        print(f"  Format: PER-SCAN keys ({len(format_a_keys)} scans)")
        scan_keys = format_a_keys
        scan_dict = {k: raw_npz[k].astype(np.float32) for k in scan_keys}
    else:
        # Format B: batch arrays (embeddings + patient_ids + timepoints)
        print(f"  Format: BATCH arrays")
        emb_key = next((k for k in all_npz_keys if raw_npz[k].ndim == 2), None)
        pid_key = next((k for k in all_npz_keys if 'patient' in k.lower() or 'pid' in k.lower()), None)
        tp_key  = next((k for k in all_npz_keys if 'time' in k.lower() or k.lower() in ['tp','tps']), None)
        emb_matrix = raw_npz[emb_key].astype(np.float32)
        raw_pids = [str(s) for s in raw_npz[pid_key]] if pid_key else [f"s{i}" for i in range(len(emb_matrix))]
        raw_tps  = list(raw_npz[tp_key]) if tp_key else list(range(len(emb_matrix)))
        scan_keys = [f"{raw_pids[i]}__t{int(raw_tps[i])}" for i in range(len(emb_matrix))]
        scan_dict = {scan_keys[i]: emb_matrix[i] for i in range(len(emb_matrix))}
        print(f"  Composite keys: {len(scan_dict)} | example: {scan_keys[:2]}")

    static_npz = raw_npz  # keep reference
    dim_static = next(iter(scan_dict.values())).shape[0]
    print(f"  Embedding dim: {dim_static} | Total scans: {len(scan_dict)}")

    for key in scan_keys:
        parts = key.split("__")
        if len(parts) != 2:
            continue
        pid, tp_str = parts
        tp = int("".join(filter(str.isdigit, tp_str)) or "0")
        emb = scan_dict[key]
        if pid not in static_patient_seqs:
            static_patient_seqs[pid] = []
        static_patient_seqs[pid].append((tp, emb, key))
    for pid in static_patient_seqs:
        static_patient_seqs[pid].sort(key=lambda x: x[0])
    print(f"  Static patients: {len(static_patient_seqs)}")
else:
    print("\n  WARNING: Static hybrid embeddings not found -- will skip comparison")

# ============================================================
# LOAD TUMOR VOLUMES
# ============================================================
vol_path = find_file(["tumor_volumes.csv"])
vol_df = None
vol_lookup = {}
if vol_path:
    vol_df = pd.read_csv(vol_path)
    print(f"\nTumor volumes: {len(vol_df)} rows from {vol_path.name}")
    print(f"  Columns: {list(vol_df.columns)}")
    cols = list(vol_df.columns)
    pid_col = next((c for c in cols if any(k in c.lower() for k in
                    ['patient', 'pid', 'subject', 'case', 'id'])), None)
    tp_col  = next((c for c in cols if any(k in c.lower() for k in
                    ['time', 'visit', 'tp', 'scan', 'follow'])), None)
    vol_col = next((c for c in cols if any(k in c.lower() for k in
                    ['vol', 'size', 'mm3', 'tumor', 'wt', 'whole', 'total'])), None)
    if vol_col is None:
        num_cols = vol_df.select_dtypes(include=['float64','float32','int64']).columns.tolist()
        vol_col = next((c for c in num_cols if c != tp_col), None)
    print(f"  Detected: pid_col={pid_col} | tp_col={tp_col} | vol_col={vol_col}")
    if pid_col and vol_col:
        for i, row in vol_df.iterrows():
            pid_val = str(row[pid_col])
            tp_val  = int(row[tp_col]) if tp_col and not pd.isna(row[tp_col]) else i
            vol_val = float(row[vol_col]) if not pd.isna(row[vol_col]) else 0.0
            vol_lookup[(pid_val, tp_val)] = vol_val
        print(f"  Volume lookup: {len(vol_lookup)} entries")
else:
    print("\n  WARNING: tumor_volumes.csv not found -- attach brats2024-metadata dataset!")

def fast_vol_lookup(pid, tp):
    tp_int = int(str(tp).replace("t", "").replace("p", ""))
    for key_pid in [pid, pid.split("-")[-1]]:
        if (key_pid, tp_int) in vol_lookup:
            return vol_lookup[(key_pid, tp_int)]
    for (k_pid, k_tp), v in vol_lookup.items():
        if tp_int == k_tp and (pid in k_pid or k_pid in pid):
            return v
    return None

# ============================================================
# BUILD PATIENT-LEVEL LABELS (using traj_meta timepoints)
# ============================================================
# Use traj_meta for timepoints so we don't depend on static_patient_seqs
patient_labels = {}
for pid in traj_pids:
    meta = traj_meta.get(pid, {})
    tps = meta.get("timepoints", [])
    n_scans = meta.get("n_scans", len(tps))
    if len(tps) < 2:
        continue
    first_tp, last_tp = tps[0], tps[-1]
    v_first = fast_vol_lookup(pid, first_tp)
    v_last  = fast_vol_lookup(pid, last_tp)
    if v_first is not None and v_last is not None and v_first > 1.0:
        delta = (v_last - v_first) / v_first
        response = "progressive" if delta > 0.25 else "responder" if delta < -0.25 else "stable"
        patient_labels[pid] = {
            "delta_vol": delta,
            "v_first": v_first,
            "v_last": v_last,
            "response": response,
            "n_scans": n_scans,
        }

print(f"\nPatients with volume labels: {len(patient_labels)}")
if len(patient_labels) == 0:
    print("  vol_lookup empty -- is tumor_volumes.csv attached?")
    print(f"  vol_lookup size: {len(vol_lookup)} | traj_meta size: {len(traj_meta)}")
    if traj_meta:
        sample_pid = list(traj_meta.keys())[0]
        print(f"  traj_meta sample: {sample_pid} -> {traj_meta[sample_pid]}")
else:
    resp_counts = defaultdict(int)
    for v in patient_labels.values():
        resp_counts[v["response"]] += 1
    for r in ["progressive", "stable", "responder"]:
        print(f"  {r}: {resp_counts[r]}")
if patient_labels:
    resp_counts = defaultdict(int)
    for v in patient_labels.values():
        resp_counts[v["response"]] += 1
    for r in ["progressive", "stable", "responder"]:
        print(f"  {r}: {resp_counts[r]}")

# Split info
splits = defaultdict(list)
for pid in traj_pids:
    sp = traj_meta.get(pid, {}).get("split", "unknown")
    splits[sp].append(pid)
for sp in ["train", "val", "test", "unknown"]:
    if splits[sp]:
        print(f"  {sp}: {len(splits[sp])} patients")

print(f"\n{'='*60}")
print(f"  DATA LOADED SUCCESSFULLY")
print(f"{'='*60}")


In [ ]:
# ============================================================
# TEMPORAL TESTS T1-T8: TaViT vs STATIC
# ============================================================
print("=" * 60)
print("  TEMPORAL TESTS: TaViT TRAJECTORY vs STATIC EMBEDDING")
print("=" * 60)

results = {"tavit": {}, "static_hybrid": {}}

# -- Helper: compute temporal metrics for a set of patient embeddings --
def compute_temporal_metrics(patient_embs, patient_seqs_dict,
                              label="model", test_pids=None):
    """
    patient_embs:      dict pid -> embedding (TaViT: 256-D; static: list of per-scan embs)
    test_pids:         if provided, restrict evaluation to these patient IDs (avoids leakage)
    Returns dict of temporal metric values.
    """
    metrics = {}
    test_filter_str = f"test-split ({len(test_pids)} patients)" if test_pids else "all patients"

    # Determine embedding type from first element
    first_val = next(iter(patient_embs.values()), None)
    is_trajectory = not isinstance(first_val, list)

    # Collect paired data: for each patient, get (emb_distance, volume_change)
    spearman_pairs = []  # (embedding_dist, volume_change)
    kendall_pairs = []
    ordering_pass = []
    coherence_vals = []
    direction_pairs = []  # (predicted_direction, actual_direction)
    rano_pairs = []
    n_matched = 0
    n_skipped_split = 0

    for pid in sorted(patient_embs.keys()):
        # If test_pids filter provided, skip non-test patients (avoids train leakage)
        if test_pids is not None and pid not in test_pids:
            n_skipped_split += 1
            continue
        if pid not in patient_seqs_dict or pid not in patient_labels:
            continue

        seqs = patient_seqs_dict[pid]
        if len(seqs) < 2:
            continue

        pl = patient_labels[pid]
        n_matched += 1

        if is_trajectory:
            # TaViT: use trajectory embedding directly
            traj_emb = patient_embs[pid]

            # T1: correlate trajectory embedding norm with absolute volume change
            spearman_pairs.append((np.linalg.norm(traj_emb), abs(pl["delta_vol"])))

            # T8: correlate first PC of trajectory with signed volume change
            kendall_pairs.append((traj_emb, pl["delta_vol"]))

        else:
            # Static: compute pairwise distances between consecutive scans
            emb_list = patient_embs[pid]
            for i in range(len(emb_list) - 1):
                e1, tp1 = emb_list[i]
                e2, tp2 = emb_list[i + 1]
                v1 = fast_vol_lookup(pid, tp1)
                v2 = fast_vol_lookup(pid, tp2)
                if v1 is None or v2 is None or v1 < 1.0:
                    continue

                dist = np.linalg.norm(e1 - e2)
                dv = (v2 - v1) / v1
                spearman_pairs.append((dist, abs(dv)))

                # Direction: did embedding move in consistent direction with volume?
                if abs(dv) > 0.05:
                    direction_pairs.append((dist, 1.0 if dv > 0 else 0.0))

            # Coherence: consecutive embedding distances should be smooth
            if len(emb_list) >= 3:
                dists = []
                for i in range(len(emb_list) - 1):
                    dists.append(np.linalg.norm(emb_list[i][0] - emb_list[i+1][0]))
                if len(dists) >= 2:
                    cv = np.std(dists) / (np.mean(dists) + 1e-8)
                    coherence_vals.append(1.0 / (1.0 + cv))

        # RANO: detect >= 25% growth events
        for i in range(len(seqs) - 1):
            v1 = fast_vol_lookup(pid, seqs[i][0])
            v2 = fast_vol_lookup(pid, seqs[-1][0])
            if v1 and v2 and v1 > 1.0:
                grown = 1 if (v2 - v1) / v1 >= 0.25 else 0
                rano_pairs.append((pid, grown))
                break

    # -- Compute T1: Spearman --
    if len(spearman_pairs) >= 10:
        x, y = zip(*spearman_pairs)
        rho, p = spearmanr(x, y)
        metrics["T1_spearman_wt"] = rho
        print(f"  {label} T1_spearman_wt: {rho:.3f} (p={p:.4f})")
    else:
        metrics["T1_spearman_wt"] = 0.0
        print(f"  {label} T1_spearman_wt: 0.000 (not enough pairs)")

    # -- Compute T8: Kendall tau (trajectory-level) --
    if is_trajectory and len(kendall_pairs) >= 10:
        emb_arr = np.stack([p[0] for p in kendall_pairs])
        deltas = np.array([p[1] for p in kendall_pairs])
        # Use first PC of trajectory embeddings
        pca = PCA(n_components=1)
        pc1 = pca.fit_transform(emb_arr).squeeze()
        tau, p = kendalltau(pc1, deltas)
        metrics["T8_kendall_tau"] = abs(tau)
        print(f"  {label} T8_kendall_tau: {abs(tau):.3f} (p={p:.4f})")

        # Also compute Spearman on PC1
        rho_pc1, _ = spearmanr(pc1, deltas)
        metrics["T1_spearman_pc1"] = abs(rho_pc1)
        print(f"  {label} T1_spearman_pc1: {abs(rho_pc1):.3f}")
    elif not is_trajectory and len(spearman_pairs) >= 10:
        x, y = zip(*spearman_pairs)
        tau, p = kendalltau(x, y)
        metrics["T8_kendall_tau"] = abs(tau)
        print(f"  {label} T8_kendall_tau: {abs(tau):.3f}")

    # -- T5: Coherence --
    if coherence_vals:
        metrics["T5_coherence"] = np.mean(coherence_vals)
        print(f"  {label} T5_coherence: {np.mean(coherence_vals):.3f}")

    # -- T4: RANO AUC --
    if len(rano_pairs) >= 10:
        rano_pids_used = [p[0] for p in rano_pairs]
        rano_labels = np.array([p[1] for p in rano_pairs])
        if is_trajectory and rano_labels.sum() > 0 and rano_labels.sum() < len(rano_labels):
            rano_embs = np.stack([patient_embs[pid] for pid in rano_pids_used])
            X = StandardScaler().fit_transform(rano_embs)
            try:
                cv = StratifiedKFold(n_splits=min(5, int(rano_labels.sum())), shuffle=True, random_state=42)
                scores = cross_val_score(RandomForestClassifier(n_estimators=100, random_state=42),
                                        X, rano_labels, cv=cv, scoring="roc_auc")
                metrics["T4_rano_auc"] = np.mean(scores)
                print(f"  {label} T4_rano_auc: {np.mean(scores):.3f}")
            except Exception as e:
                metrics["T4_rano_auc"] = 0.5
                print(f"  {label} T4_rano_auc: 0.500 (error: {e})")

    return metrics


# ============================================================
# RUN TEMPORAL TESTS: TaViT
# ============================================================
print("\n-- TaViT Trajectory Embeddings (test-split only, n=83) --")
tavit_embs = {pid: traj_npz[pid].astype(np.float32) for pid in traj_pids}
test_pids_set = set(splits.get("test", []))
print(f"  Restricting to: {len(test_pids_set)} test patients")
results["tavit"] = compute_temporal_metrics(
    tavit_embs, static_patient_seqs, label="TaViT", test_pids=test_pids_set)

# ============================================================
# RUN TEMPORAL TESTS: Static Hybrid (for comparison)
# ============================================================
if static_npz is not None:
    # Static: use test-split only for fair comparison
    print("\n-- Static Hybrid Embeddings (test-split only, n=83) --")
    static_embs = {}
    for pid in test_pids_set:
        if pid in static_patient_seqs:
            seqs = static_patient_seqs[pid]
            static_embs[pid] = [(s[1], s[0]) for s in seqs]
    print(f"  Static patients found: {len(static_embs)}")
    results["static_hybrid"] = compute_temporal_metrics(
        static_embs, static_patient_seqs, label="Static")


In [ ]:
# ============================================================
# TRAJECTORY-SPECIFIC TESTS (Test-Split Only, n=83)
# ============================================================
from collections import defaultdict
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score, classification_report
from sklearn.model_selection import cross_val_score
from scipy.stats import mannwhitneyu

test_pids_eval = set(splits.get("test", []))
labeled_pids = [pid for pid in traj_pids if pid in patient_labels]
labeled_test_pids = [pid for pid in labeled_pids if pid in test_pids_eval]
print(f"  Test patients with labels: {len(labeled_test_pids)}")

eval_pids = labeled_test_pids
if len(eval_pids) < 20:
    print(f"  WARNING: test split too small ({len(eval_pids)}), using all {len(labeled_pids)}")
    eval_pids = labeled_pids

if len(eval_pids) >= 10:
    X_traj = np.stack([traj_npz[pid].astype(np.float32) for pid in eval_pids])
    y_delta = np.array([patient_labels[pid]["delta_vol"] for pid in eval_pids])
    y_response = np.array([patient_labels[pid]["response"] for pid in eval_pids])
    X_scaled = StandardScaler().fit_transform(X_traj)

    # ============================================================
    # N9-RATIO: Physics-Informed Magnitude via Ratio Head
    # ============================================================
    print("\n  N9-RATIO: Physics-Informed Magnitude Prediction")
    print("       ratio_head(frozen CLS) × known V_baseline")
    try:
        import torch
        import torch.nn as nn
        from sklearn.isotonic import IsotonicRegression
        from sklearn.metrics import r2_score
        from scipy.stats import spearmanr as _spearmanr

        train_pids = [p for p in splits.get("train", []) if p in patient_labels and p in traj_npz.files]
        X_train_r = torch.tensor(np.stack([traj_npz[p].astype(np.float32) for p in train_pids]))
        X_test_r  = torch.tensor(np.stack([traj_npz[p].astype(np.float32) for p in eval_pids]))

        def safe_log_ratio(pid):
            vf = patient_labels[pid]["v_first"]
            vl = patient_labels[pid]["v_last"]
            return float(np.log(max(vl, 1.0) / max(vf, 1.0)))

        y_train_ratio = torch.tensor([safe_log_ratio(p) for p in train_pids], dtype=torch.float32)
        y_test_ratio  = np.array([safe_log_ratio(p) for p in eval_pids])

        v_first_test = np.array([patient_labels[p]["v_first"] for p in eval_pids])
        v_last_test  = np.array([patient_labels[p]["v_last"] for p in eval_pids])
        true_delta_mm3 = v_last_test - v_first_test
        true_rel = (v_last_test - v_first_test) / np.maximum(v_first_test, 1.0)

        scaler_r = StandardScaler()
        X_train_s = torch.tensor(scaler_r.fit_transform(X_train_r.numpy()), dtype=torch.float32)
        X_test_s  = torch.tensor(scaler_r.transform(X_test_r.numpy()), dtype=torch.float32)

        torch.manual_seed(42)  # reproducible ratio head across runs
        np.random.seed(42)
        ratio_head = nn.Linear(256, 1)
        nn.init.xavier_uniform_(ratio_head.weight)
        nn.init.zeros_(ratio_head.bias)
        opt = torch.optim.Adam(ratio_head.parameters(), lr=1e-3, weight_decay=1e-4)

        ratio_head.train()
        for ep in range(80):
            pred = ratio_head(X_train_s).squeeze(-1)
            loss = nn.functional.mse_loss(pred, y_train_ratio)
            opt.zero_grad(); loss.backward(); opt.step()

        ratio_head.eval()
        with torch.no_grad():
            train_pred_log = ratio_head(X_train_s).squeeze(-1).numpy()
            test_pred_log  = ratio_head(X_test_s).squeeze(-1).numpy()

        # Isotonic calibration
        iso = IsotonicRegression(out_of_bounds="clip")
        iso.fit(train_pred_log, y_train_ratio.numpy())
        test_pred_cal = iso.predict(test_pred_log)

        pred_ratio_cal = np.exp(test_pred_cal) - 1.0
        pred_delta_cal = v_first_test * pred_ratio_cal

        rho_raw, p_raw = _spearmanr(true_delta_mm3, v_first_test * (np.exp(test_pred_log) - 1.0))
        rho_cal, p_cal = _spearmanr(true_delta_mm3, pred_delta_cal)
        r2_rel_cal = r2_score(true_rel, pred_ratio_cal)

        print(f"    Trained on {len(train_pids)} patients | Tested on {len(eval_pids)}")
        print(f"    Parameters: 257 (frozen backbone)")
        print(f"    Spearman ρ (magnitude):  {rho_cal:.3f} (p={p_cal:.4f})")
        print(f"    R² (relative ratio):     {r2_rel_cal:.3f}")
        results.setdefault("tavit", {})["N9_ratio_spearman"] = round(float(rho_cal), 4)
        results.setdefault("tavit", {})["N9_ratio_R2_rel"]   = round(float(r2_rel_cal), 4)
    except Exception as e:
        import traceback
        print(f"    Error: {e}"); traceback.print_exc()

    # ============================================================
    # N10: Response Classification (3-class CV)
    # ============================================================
    print(f"\n  N10: Response Classification")
    resp_counts = defaultdict(int)
    for r in y_response: resp_counts[r] += 1
    print(f"    Classes: {dict(resp_counts)}")
    try:
        n_cv = min(5, max(3, len(eval_pids) // 10))
        mask = np.array([r in ["progressive", "stable"] for r in y_response])
        X_filt = X_scaled[mask]; y_filt = y_response[mask]

        if len(set(y_filt)) >= 2:
            clf = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42)
            f1_cv = cross_val_score(clf, X_filt, y_filt, cv=n_cv, scoring="f1_weighted")
            print(f"    CV Weighted F1:  {f1_cv.mean():.3f} ± {f1_cv.std():.3f}")
            results.setdefault("tavit", {})["N10_response_F1"] = round(float(f1_cv.mean()), 4)
    except Exception as e:
        print(f"    Error: {e}")

    # ============================================================
    # N11: Trajectory Clustering
    # ============================================================
    print(f"\n  N11: Trajectory Clustering")
    for k in [2, 3]:
        km = KMeans(n_clusters=k, random_state=42, n_init=10)
        clust = km.fit_predict(X_scaled)
        sil = silhouette_score(X_scaled, clust)
        ari = adjusted_rand_score(y_response, clust)
        print(f"    K={k}: Silhouette={sil:.3f} | ARI={ari:.3f}")
        results.setdefault("tavit", {})[f"N11_silhouette_k{k}"] = round(sil, 4)
        results.setdefault("tavit", {})[f"N11_ari_k{k}"] = round(ari, 4)

    # ============================================================
    # N12: Progressive vs Stable Separation (Cohen's d)
    # ============================================================
    print(f"\n  N12: Progressive vs Stable Separation")
    prog_mask = y_response == "progressive"
    stab_mask = y_response == "stable"
    if prog_mask.sum() >= 5 and stab_mask.sum() >= 5:
        norms = np.linalg.norm(X_traj, axis=1)
        d_prog = norms[prog_mask]; d_stab = norms[stab_mask]
        pooled_std = np.sqrt((d_prog.var() + d_stab.var()) / 2)
        cohens_d = abs(d_prog.mean() - d_stab.mean()) / max(pooled_std, 1e-8)
        _, mw_p = mannwhitneyu(d_prog, d_stab, alternative="two-sided")
        print(f"    Cohen's d = {cohens_d:.3f} (p={mw_p:.4f})")
        results.setdefault("tavit", {})["N12_cohens_d"] = round(cohens_d, 4)

    # ============================================================
    # N13: Velocity (documented limitation — synthetic timestamps)
    # ============================================================
    print(f"\n  N13: Velocity — EXPECTED FAIL (synthetic 90-day intervals)")
    print(f"    BraTS lacks real timestamps; velocity = delta / fake_time")
    n_scans = np.array([patient_labels[pid].get("n_scans", 2) for pid in eval_pids])
    velocity = y_delta / (n_scans - 1 + 1e-8)
    try:
        from sklearn.ensemble import RandomForestRegressor
        rf_vel = RandomForestRegressor(n_estimators=200, random_state=42)
        vel_r2 = cross_val_score(rf_vel, X_scaled, velocity, cv=5, scoring="r2")
        print(f"    Velocity R² (RF): {vel_r2.mean():.3f} ± {vel_r2.std():.3f}")
        results.setdefault("tavit", {})["N13_velocity_R2"] = round(float(vel_r2.mean()), 4)
    except Exception as e:
        print(f"    Error: {e}")
else:
    print(f"  Not enough labeled test patients ({len(eval_pids)})")


In [ ]:
# ============================================================
# FINAL SUMMARY
# ============================================================
print("\n" + "=" * 60)
print("  TaViT EVALUATION — FINAL RESULTS")
print("=" * 60)

# Static baselines
static_t1 = results.get("static", {}).get("T1_spearman_wt", 0.462)
static_t8 = results.get("static", {}).get("T8_kendall_tau", 0.321)
static_t4 = results.get("static", {}).get("T4_rano_auc", 0.603)

tavit = results.get("tavit", {})
t1_pc1 = tavit.get("T1_spearman_pc1", 0)
t8 = tavit.get("T8_kendall_tau", 0)
t4 = tavit.get("T4_rano_auc", 0)

print(f"\n  ── Temporal Direction (TaViT vs Static Baseline) ──")
print(f"  {'Metric':<25} {'Static':>8} {'TaViT':>8} {'Δ':>8}")
print(f"  {'-'*55}")
print(f"  {'T1 Spearman (PC1)':<25} {static_t1:>8.3f} {t1_pc1:>8.3f} {(t1_pc1/static_t1-1)*100:>+7.0f}%")
print(f"  {'T8 Kendall τ':<25} {static_t8:>8.3f} {t8:>8.3f} {(t8/static_t8-1)*100:>+7.0f}%")
print(f"  {'T4 RANO AUC':<25} {static_t4:>8.3f} {t4:>8.3f} {(t4/static_t4-1)*100:>+7.0f}%")

print(f"\n  ── Magnitude & Classification ──")
print(f"  N9-RATIO Spearman ρ:   {tavit.get('N9_ratio_spearman', float('nan')):.3f}")
print(f"  N9-RATIO R² (relative): {tavit.get('N9_ratio_R2_rel', float('nan')):.3f}")
print(f"  N10 Response F1 (CV):   {tavit.get('N10_response_F1', float('nan')):.3f}")
print(f"  N12 Cohen's d:          {tavit.get('N12_cohens_d', float('nan')):.3f}")

print(f"\n  ── Clustering ──")
print(f"  N11 Silhouette (K=2):   {tavit.get('N11_silhouette_k2', float('nan')):.3f}")
print(f"  N11 Silhouette (K=3):   {tavit.get('N11_silhouette_k3', float('nan')):.3f}")

print(f"\n  ── Known Limitations ──")
print(f"  N13 Velocity R²:        {tavit.get('N13_velocity_R2', float('nan')):.3f}  (synthetic timestamps)")

# Pass/fail
print(f"\n{'='*60}")
print(f"  PASS/FAIL")
print(f"{'='*60}")
checks = [
    ("T1 Spearman PC1",        t1_pc1,                                     0.3,  False),
    ("T8 Kendall τ",            t8,                                        0.3,  False),
    ("T4 RANO AUC",             t4,                                        0.65, False),
    ("N9-RATIO Spearman",       tavit.get("N9_ratio_spearman", 0),         0.4,  False),
    ("N10 Response F1",         tavit.get("N10_response_F1", 0),           0.5,  False),
    ("N12 Cohen's d",           tavit.get("N12_cohens_d", 0),              0.5,  False),
    ("N13 Velocity R²",         tavit.get("N13_velocity_R2", -1),          0.1,  True),
]
n_pass = n_fail = 0
for name, val, thresh, is_r2 in checks:
    passed = (val >= thresh) if is_r2 else (abs(val) >= thresh)
    status = "✓ PASS" if passed else "✗ FAIL"
    if passed: n_pass += 1
    else: n_fail += 1
    note = " (synthetic timestamps)" if "N13" in name and not passed else ""
    print(f"  {name:<25} {val:>8.3f}  ≥ {thresh:<5}  {status}{note}")

print(f"\n  Score: {n_pass}/{n_pass+n_fail} PASS | {n_fail}/{n_pass+n_fail} FAIL")

# Improvement report
print(f"\n{'='*60}")
print(f"  TaViT vs STATIC — IMPROVEMENT REPORT")
print(f"{'='*60}")
for name, static, tavit_v in [("Temporal correlation", static_t1, t1_pc1),
                                ("Trajectory ordering", static_t8, t8),
                                ("RANO detection", static_t4, t4)]:
    pct = (tavit_v / max(static, 1e-6) - 1) * 100
    tag = "IMPROVED" if pct > 5 else ("same" if pct > -5 else "worse")
    print(f"  {name:<25} static={static:.3f} → TaViT={tavit_v:.3f}  ({pct:+.0f}%)  {tag}")

print(f"\n  All outputs: {OUTPUT_ROOT}")
print(f"  Phase 4A complete.")


In [ ]:
# ============================================================
# VISUALIZATIONS
# ============================================================
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from sklearn.manifold import TSNE
    HAS_MPL = True
except ImportError:
    HAS_MPL = False
    print("  matplotlib not available -- skipping plots")

if HAS_MPL and len(patient_labels) >= 20:
    labeled_pids_vis = [pid for pid in traj_pids if pid in patient_labels]
    X_vis = np.stack([traj_npz[pid].astype(np.float32) for pid in labeled_pids_vis])
    y_resp = np.array([patient_labels[pid]["response"] for pid in labeled_pids_vis])
    y_delta = np.array([patient_labels[pid]["delta_vol"] for pid in labeled_pids_vis])

    # ---- t-SNE colored by response ----
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(X_vis)-1))
    X_2d = tsne.fit_transform(X_vis)

    color_map = {"progressive": "#e74c3c", "stable": "#3498db", "responder": "#2ecc71"}
    for resp, color in color_map.items():
        mask = y_resp == resp
        if mask.sum() > 0:
            axes[0].scatter(X_2d[mask, 0], X_2d[mask, 1], c=color, label=resp,
                          alpha=0.6, s=30, edgecolors="white", linewidths=0.3)
    axes[0].set_title("TaViT Trajectories -- by Response", fontsize=13)
    axes[0].legend()
    axes[0].set_xlabel("t-SNE 1")
    axes[0].set_ylabel("t-SNE 2")

    # ---- t-SNE colored by volume change (continuous) ----
    sc = axes[1].scatter(X_2d[:, 0], X_2d[:, 1], c=np.clip(y_delta, -2, 2),
                        cmap="RdYlGn_r", alpha=0.6, s=30, edgecolors="white",
                        linewidths=0.3)
    plt.colorbar(sc, ax=axes[1], label="Volume Change (fraction)")
    axes[1].set_title("TaViT Trajectories -- by Volume Change", fontsize=13)
    axes[1].set_xlabel("t-SNE 1")
    axes[1].set_ylabel("t-SNE 2")

    plt.tight_layout()
    plt.savefig(FIG_DIR / "tavit_tsne_response.png", dpi=150, bbox_inches="tight")
    print(f"  Saved: {FIG_DIR / 'tavit_tsne_response.png'}")
    plt.close()

    # ---- PCA: PC1 vs volume change scatter ----
    fig, ax = plt.subplots(figsize=(8, 6))
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X_vis)
    sc = ax.scatter(X_pca[:, 0], y_delta, c=[color_map.get(r, "gray") for r in y_resp],
                   alpha=0.5, s=20)
    ax.set_xlabel(f"PC1 (explained var: {pca.explained_variance_ratio_[0]:.1%})")
    ax.set_ylabel("Volume Change (fraction)")
    ax.set_title("TaViT PC1 vs Volume Change")
    ax.axhline(y=0, color="gray", linestyle="--", alpha=0.3)
    ax.axhline(y=0.25, color="red", linestyle="--", alpha=0.3, label="RANO +25%")
    ax.axhline(y=-0.25, color="green", linestyle="--", alpha=0.3, label="RANO -25%")
    ax.legend()
    plt.tight_layout()
    plt.savefig(FIG_DIR / "tavit_pc1_vs_volume.png", dpi=150, bbox_inches="tight")
    print(f"  Saved: {FIG_DIR / 'tavit_pc1_vs_volume.png'}")
    plt.close()

    # ---- Embedding norm distribution by response ----
    fig, ax = plt.subplots(figsize=(8, 5))
    for resp, color in color_map.items():
        mask = y_resp == resp
        if mask.sum() > 0:
            norms = np.linalg.norm(X_vis[mask], axis=1)
            ax.hist(norms, bins=20, alpha=0.5, color=color, label=resp)
    ax.set_xlabel("Trajectory Embedding Norm")
    ax.set_ylabel("Count")
    ax.set_title("Embedding Norm Distribution by Response")
    ax.legend()
    plt.tight_layout()
    plt.savefig(FIG_DIR / "tavit_norm_by_response.png", dpi=150, bbox_inches="tight")
    print(f"  Saved: {FIG_DIR / 'tavit_norm_by_response.png'}")
    plt.close()

    print(f"  All figures in: {FIG_DIR}")


In [ ]:
# ============================================================
# SAVE RESULTS JSON
# ============================================================
import json as _json2

with open(OUTPUT_ROOT / "tavit_eval_results.json", "w") as f:
    _json2.dump(results, f, indent=2, default=str)
print(f"  Results saved: {OUTPUT_ROOT / 'tavit_eval_results.json'}")
print(f"  Figures saved: {FIG_DIR}")
print(f"\n  Phase 4A complete. Ready for Phase 5 (LLM Narrative Generation).")
